# पाठ 11 - मोडल कन्टेक्स्ट प्रोटोकल (MCP)

The **मोडल कन्टेक्स्ट प्रोटोकल (MCP)** एउटा खुला मानक हो जसले एजेन्टहरूलाई रनटाइममा गतिशील रूपमा उपकरणहरू, स्रोतहरू, र डाटा स्रोतहरू पत्ता लगाउन र प्रयोग गर्न सक्षम बनाउँछ। एजेन्टमा उपकरणहरू हार्डकोड गर्ने सट्टा, MCP ले एजेन्टहरूलाई आवश्यक परेमा क्षमताहरू प्रदर्शन गर्ने बाह्य सर्भरहरूसँग जडान हुन दिन्छ।

यस पाठमा, तपाईंले सिक्नुहुनेछ:
- MCP के हो र एजेन्ट प्रणालीहरूका लागि किन महत्त्वपूर्ण छ
- MCP क्लाइन्ट-सर्भर आर्किटेक्चर कसरी काम गर्छ
- MCP-शैलीको उपकरण पत्ता लगाउने प्रविधि प्रयोग गर्ने एजेन्ट कसरी बनाउने


## सेटअप

**पूर्वआवश्यकताहरू:**
- Azure AI Foundry परियोजना जसमा परिनियोजित मोडेल छ
- प्रमाणीकरणका लागि `az login` चलाउनुहोस्


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Model Context Protocol (MCP) भनेको के हो?

MCP ले AI एजेन्टहरूले बाह्य उपकरणहरू र डाटा स्रोतहरू पत्ता लगाउन र तिनसँग अन्तरक्रिया गर्न प्रयोग गर्ने मानक तरिका परिभाषित गर्छ:

- **MCP Server**: मानक प्रोटोकल मार्फत उपकरणहरू, स्रोतहरू, र प्रम्प्टहरू उपलब्ध गराउँछ
- **MCP Client**: सर्भरहरूमा जडान हुने र उपलब्ध क्षमताहरू पत्ता लगाउने एजेन्ट रनटाइम
- **Dynamic Discovery**: एजेन्टहरूले हार्डकोड गरिएको उपकरणहरू आवश्यक छैन — तिनीहरूले रनटाइममा के उपलब्ध छ पत्ता लगाउँछन्

यो विस्तारयोग्य एजेन्ट प्रणालीहरू निर्माण गर्न शक्तिशाली छ जहाँ नयाँ क्षमताहरू एजेन्टको कोड संशोधन नगरी थप्न सकिन्छ।


## MCP कसरी काम गर्छ

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. एजेन्ट (MCP client) एक MCP server सँग जडान हुन्छ
2. सर्भर उपलब्ध उपकरणहरू र तिनीहरूको स्किमाहरूको सूचीसहित जवाफ दिन्छ
3. एजेन्ट त्यसपछि आफ्नो तर्क प्रक्रियाको क्रममा पत्ता लागेको कुनै पनि उपकरणलाई कल गर्न सक्छ
4. नतिजाहरू उही प्रोटोकल मार्फत फिर्ता आउँछन्


## MCP उपकरण पत्ता लगाउने अनुकरण

वास्तविक MCP सर्भरले चलिरहेको सर्भर प्रक्रिया आवश्यक पर्ने हुँदा, हामी त्यो ढाँचालाई `@tool` फंक्शनहरू प्रयोग गरेर प्रदर्शन गर्नेछौं जसले MCP-सम्पर्कित आवास सेवा के प्रदान गर्ला भन्ने अनुकरण गर्छ।

उत्पादन वातावरणमा, यी उपकरणहरू स्थानीय रूपमा परिभाषित गरिएको भन्दा MCP सर्भरबाट गतिशील रूपमा पत्ता लगाइने थिए।


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## MCP-शैली उपकरणहरूसँग एक एजेन्ट निर्माण


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP उत्पादनमा

उत्पादन वातावरणमा, MCP ले शक्तिशाली ढाँचाहरू सक्षम गर्दछ:

- **गतिशील उपकरण पत्ता लगाउने**: एजेन्टहरूले MCP सर्भरहरूमा जडान गरी रनटाइममा उपकरणहरू पत्ता लगाउँछन्
- **विच्छेदित वास्तुकला**: उपकरण प्रदायकहरूले एजेन्टबाट स्वतन्त्र रूपमा अद्यावधिक गर्न सक्छन्
- **संगठनहरूबीच साझेदारी**: टोलीहरूले MCP सर्भरहरू मार्फत त्यस्ता क्षमताहरू उपलब्ध गराउन सक्छन् जुन कुनै पनि एजेन्टले प्रयोग गर्न सक्छ
- **Microsoft Agent Framework समर्थन**: MAF सँग `mcp` एकीकरणमार्फत निर्मित MCP क्लाइन्ट समर्थन छ

MAF सँग वास्तविक MCP सर्भर प्रयोग गर्नका लागि, तपाईं `hosted_mcp_tool()` वा MCP क्लाइन्ट एकीकरणमार्फत जडान गर्नुहुनेछ।

**थप जान्नुहोस्:**
- [MCP विशिष्टता](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework को MCP समर्थन](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## सारांश

यस पाठमा, तपाईंले सिक्नुभयो:
- **MCP** एजेन्टहरू र उपकरण प्रदायकहरूबीच गतिशील उपकरण पत्ता लगाउने एक खुला मापदण्ड हो
- **client-server architecture** ले एजेन्टहरूलाई रनटाइममा क्षमताहरू पत्ता लगाउन दिन्छ
- MCP ले **विस्तारयोग्य, अलग गरिएका एजेन्ट प्रणालीहरू** सक्षम बनाउँछ जहाँ उपकरणहरूलाई कोड परिवर्तन बिना थप्न सकिन्छ
- Microsoft Agent Framework ले उत्पादन प्रयोगका लागि **अंगभूत MCP समर्थन** प्रदान गर्दछ


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
अस्वीकरण:
यो दस्तावेज एआई अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरी अनुवाद गरिएको हो। हामी शुद्धताको लागि प्रयास गर्छौं, तर कृपया ध्यान दिनुहोस् कि स्वचालित अनुवादहरूमा त्रुटि वा अशुद्धता हुन सक्छ। मूल भाषामा रहेको मूल दस्तावेजलाई प्रामाणिक स्रोत मानिनु पर्छ। महत्वपूर्ण जानकारीका लागि पेशेवर मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलतफहमी वा भ्रामक व्याख्याका लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
